In [159]:
import pandas as pd


In [160]:
data = pd.read_csv("../data/SMSSpamCollection.txt",sep= '\t',names = ["label","text"])

In [161]:
data.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [162]:
from nltk.stem import PorterStemmer,WordNetLemmatizer
from nltk.corpus import stopwords
lem = WordNetLemmatizer()


In [163]:
data['text'] = data['text'].str.lower()

In [164]:
from nltk.tokenize import word_tokenize
data['text']  = data['text'].apply(word_tokenize)

In [165]:
data['text'].shape

(5572,)

In [166]:
inputs = data['text']

In [167]:
inputs.head()

0    [go, until, jurong, point, ,, crazy, .., avail...
1             [ok, lar, ..., joking, wif, u, oni, ...]
2    [free, entry, in, 2, a, wkly, comp, to, win, f...
3    [u, dun, say, so, early, hor, ..., u, c, alrea...
4    [nah, i, do, n't, think, he, goes, to, usf, ,,...
Name: text, dtype: object

In [168]:
inputs = inputs.apply(lambda x : [word for word in x if word not in stopwords.words('english')])

In [169]:
import re
inputs = inputs.apply(lambda x : [re.sub("[^a-zA-Z]"," ",word) for word in x] )

In [ ]:
inputs = input

0       [go, jurong, point,  , crazy,   , available, b...
1                [ok, lar,    , joking, wif, u, oni,    ]
2       [free, entry,  , wkly, comp, win, fa, cup, fin...
3       [u, dun, say, early, hor,    , u, c, already, ...
4       [nah, n t, think, goes, usf,  , lives, around,...
                              ...                        
5567    [ nd, time, tried,  , contact, u , u,     , po...
5568                [ , b, going, esplanade, fr, home,  ]
5569           [pity,  ,  , mood,  ,    , suggestions,  ]
5570    [guy, bitching, acted, like,  d, interested, b...
5571                                [rofl,  , true, name]
Name: text, Length: 5572, dtype: object

In [177]:
inputs = inputs.apply(lambda x : lem.lemmatize(x) if x != ' ' else x )

In [178]:
inputs

0       go jurong point   crazy    available bugis n g...
1                         ok lar     joking wif u oni    
2       free entry   wkly comp win fa cup final tkts  ...
3             u dun say early hor     u c already say    
4            nah n t think goes usf   lives around though
                              ...                        
5567     nd time tried   contact u  u      pound prize...
5568                          b going esplanade fr home  
5569                    pity     mood       suggestions  
5570    guy bitching acted like  d interested buying s...
5571                                     rofl   true name
Name: text, Length: 5572, dtype: object

In [173]:
inputs = inputs.apply(lambda x : ' '.join(x))

In [175]:
inputs

0       go jurong point   crazy    available bugis n g...
1                         ok lar     joking wif u oni    
2       free entry   wkly comp win fa cup final tkts  ...
3             u dun say early hor     u c already say    
4            nah n t think goes usf   lives around though
                              ...                        
5567     nd time tried   contact u  u      pound prize...
5568                          b going esplanade fr home  
5569                    pity     mood       suggestions  
5570    guy bitching acted like  d interested buying s...
5571                                     rofl   true name
Name: text, Length: 5572, dtype: object

In [201]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure required NLTK packages are downloaded
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

# Load the dataset
file_path = "../data/SMSSpamCollection.txt"
df = pd.read_csv(file_path, sep='\t', header=None, names=['label', 'message'])

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Text preprocessing function
def preprocess(text):
    # Lowercase
    text = text.lower()
    text = re.sub("[^a-zA-Z]",' ',text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    words = nltk.word_tokenize(text)
    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)


# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess)

# Show a sample
print(df[['label', 'clean_message']].head())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ravik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ravik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ravik\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


  label                                      clean_message
0   ham  go jurong point crazy available bugis n great ...
1   ham                            ok lar joking wif u oni
2  spam  free entry wkly comp win fa cup final tkts st ...
3   ham                u dun say early hor u c already say
4   ham                nah think go usf life around though


In [202]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500)


In [203]:
x  = cv.fit_transform(df['clean_message'])

In [204]:
x.shape

(5572, 2500)

In [205]:
new_df = pd.DataFrame(x.toarray(),columns = cv.get_feature_names_out())

In [235]:
y=data['label'].apply(lambda x : 1 if x=='spam' else 0)

In [ ]:
# import numpy as np

# y=np.array(y).reshape(-1,1)

In [236]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(new_df,y,random_state=42)

In [237]:
from sklearn.naive_bayes import MultinomialNB

In [238]:
nb = MultinomialNB()

In [240]:
nb.fit(x_train,y_train)

MultinomialNB()

In [241]:
pred = nb.predict(x_test)

In [242]:
from sklearn.metrics import accuracy_score,confusion_matrix
acc = accuracy_score(y_test,pred)
conf = confusion_matrix(y_test,pred)

In [243]:
acc

0.9784637473079684

In [244]:
conf

array([[1187,   20],
       [  10,  176]], dtype=int64)

In [246]:
y_test.value_counts()

label
0    1207
1     186
Name: count, dtype: int64